# Cost-Sensitive Credit Default Prediction & Decision Threshold Optimization
**ES 335 Machine Learning Project** — Tagalpallewar Abhishek Prashant (24110364) & Siddhantkumar Vinay Thute (24110373)

This notebook provides an end-to-end, reproducible walkthrough of our leak-free pipeline, model comparison across eight classifiers, probability calibration, and Bayesian decision threshold optimization under asymmetric financial costs ($C_{\text{FN}} = \$10{,}000$, $C_{\text{FP}} = \$1{,}500$).

In [ ]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Image, display

sys.path.append('../src')
from preprocessing import build_preprocessor

train_df = pd.read_csv('../data/train.csv')
val_df = pd.read_csv('../data/val.csv')
test_df = pd.read_csv('../data/test.csv')

print(f'Train shape: {train_df.shape} | Default rate: {train_df["TARGET"].mean():.2%}')
print(f'Val shape  : {val_df.shape} | Default rate: {val_df["TARGET"].mean():.2%}')
print(f'Test shape : {test_df.shape} | Default rate: {test_df["TARGET"].mean():.2%}')

## 1. Leak-Free Feature Engineering & Preprocessing
`build_preprocessor()` encapsulates row-wise financial ratio construction (`CreditFeatureEngineer`) followed by `StandardScaler` and `OneHotEncoder(drop='first')` fitted strictly on training folds.

In [ ]:
prep = build_preprocessor()
X_train_proc = prep.fit_transform(train_df.drop(columns=['TARGET']))
feature_names = prep.named_steps['preprocessing'].get_feature_names_out()
print(f'Engineered feature matrix shape: {X_train_proc.shape}')
print('Sample features:', list(feature_names[:10]))

## 2. Validation Leaderboard Across All Eight Models
Models are ranked by expected validation dollar loss at the theoretical Bayesian threshold $\tau^* = \frac{1500}{1500 + 10000} \approx 0.1304$.

In [ ]:
val_leaderboard = pd.read_csv('../results/val_leaderboard.csv')
display(val_leaderboard)

## 3. Holdout Test Set Performance (Evaluated Once)
Based on validation loss at $\tau^* = 0.1304$ and a validation threshold sweep selecting $\tau = 0.135$, **HistGradientBoosting** was evaluated once on the isolated 4,500-client test split.

In [ ]:
with open('../results/test_metrics.json', 'r') as f:
    test_metrics = json.load(f)
pd.Series(test_metrics)

## 4. Key Diagnostic Visualizations

In [ ]:
for img_name in ['class_imbalance.png', 'feature_correlation.png', 'calibration_curves.png', 'final_cost_curve.png']:
    img_path = Path('../docs/assets') / img_name
    if img_path.exists():
        print(img_name)
        display(Image(filename=str(img_path), width=700))